# 02 — Cleaning & Preprocessing

**Phase 3** (see `CONTINUITY.md`). This notebook walks through what `src/data_prep.py` does, step by step. The production code is the `.py` file; this notebook explains and verifies it.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.load_data import load_raw
from src.data_prep import clean, split, build_preprocessor, prepare

raw = load_raw()
print(raw.shape)

(7043, 21)


## 3.1 Fix `TotalCharges`
It is stored as text. 11 rows are blank — all are customers with `tenure = 0` who have never been billed, so the correct value is **0**, not the mean or median.

In [2]:
blank = raw["TotalCharges"].str.strip() == ""
print("Blank TotalCharges:", blank.sum())
print(raw.loc[blank, ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head())

df = clean(raw)
print("\nAfter clean():", df["TotalCharges"].dtype, "| NaN:", df["TotalCharges"].isna().sum(),
      "| zero-tenure rows now:", df.loc[df.tenure == 0, "TotalCharges"].unique())

Blank TotalCharges: 11
      tenure  MonthlyCharges TotalCharges Churn
488        0           52.55                 No
753        0           20.25                 No
936        0           80.85                 No
1082       0           25.75                 No
1340       0           56.05                 No

After clean(): float64 | NaN: 0 | zero-tenure rows now: [0.]


## 3.2 Drop the ID, encode the target

In [3]:
print("customerID dropped:", "customerID" not in df.columns)
print(df["Churn"].value_counts())

customerID dropped: True
Churn
0    5174
1    1869
Name: count, dtype: int64


## 3.4 Stratified 80/20 split
Stratifying keeps the churn rate identical in train and test (it matters with a 26.5% minority class).

In [4]:
X_train, X_test, y_train, y_test = split(df)
print(f"Train {X_train.shape}, churn {y_train.mean():.2%}")
print(f"Test  {X_test.shape}, churn {y_test.mean():.2%}")

Train (5634, 19), churn 26.54%
Test  (1409, 19), churn 26.54%


## 3.3 + 3.5 Encode and scale inside a `ColumnTransformer`
- **Numeric** (`tenure`, `MonthlyCharges`, `TotalCharges`) → `StandardScaler`
- **Categorical** (15 cols) → `OneHotEncoder(drop="if_binary")`: Yes/No columns become one 0/1 column; multi-level columns get one column per level
- **SeniorCitizen** → already 0/1, passed through

The transformer is **fitted on the training set only**, then applied to the test set — no information leaks from test into training.

In [5]:
pre = build_preprocessor().set_output(transform="pandas")
Xt_train = pre.fit_transform(X_train)
Xt_test = pre.transform(X_test)
print("Features after encoding:", Xt_train.shape[1])
print("Any NaN?", Xt_train.isna().any().any())
print(Xt_train.iloc[:5, :8].round(2).to_string())

Features after encoding: 40
Any NaN? False
      tenure  MonthlyCharges  TotalCharges  gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  MultipleLines_No
3738    0.10           -0.52         -0.26          1.0          0.0             0.0               0.0               0.0
3151   -0.71            0.34         -0.50          1.0          1.0             1.0               1.0               1.0
4860   -0.79           -0.81         -0.75          1.0          1.0             1.0               0.0               0.0
3867   -0.26            0.28         -0.17          0.0          1.0             0.0               1.0               1.0
3810   -1.28           -0.68         -0.99          1.0          1.0             1.0               1.0               1.0


In [6]:
# Scaled numerics: mean ~0, std ~1 on train
print(Xt_train[["tenure", "MonthlyCharges", "TotalCharges"]].describe().loc[["mean", "std"]].round(3))

      tenure  MonthlyCharges  TotalCharges
mean    -0.0            -0.0           0.0
std      1.0             1.0           1.0


## One call does it all
`prepare()` runs every step above. In Phase 5, `prepare(fit=False)` returns the **unfitted** preprocessor so it can sit inside a model `Pipeline` and be refit within each cross-validation fold.

In [7]:
X_train, X_test, y_train, y_test, pre = prepare()
print(X_train.shape, X_test.shape)

(5634, 40) (1409, 40)
